# Food Delivery Business Analytics — Exploratory Data Analysis

This notebook explores the raw dataset, validates data quality issues, and visualizes the key patterns that motivate the SQL KPI queries and Power BI dashboard.

**Sections**
1. Load & inspect raw data
2. Data quality checks (nulls, bad values, duplicates)
3. Revenue trends
4. Customer behavior
5. Delivery operations
6. Key takeaways

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

orders = pd.read_csv('../data/raw/orders.csv', parse_dates=['order_timestamp'])
customers = pd.read_csv('../data/raw/customers.csv', parse_dates=['signup_date'])
restaurants = pd.read_csv('../data/raw/restaurants.csv', parse_dates=['onboarded_date'])
partners = pd.read_csv('../data/raw/delivery_partners.csv', parse_dates=['joined_date'])

print(orders.shape, customers.shape, restaurants.shape, partners.shape)
orders.head()

## 2. Data Quality Checks

In [ ]:
print('Missing values per column:')
print(orders.isna().sum()[orders.isna().sum() > 0])

print('\nInvalid delivery times (sentinel -1):', (orders['actual_delivery_min'] < 0).sum())
print('Duplicate order_id rows:', orders['order_id'].duplicated().sum())
print('\nOrder value range:', orders['order_value'].min(), '-', orders['order_value'].max())

In [ ]:
# Apply the same cleaning as src/data_cleaning.py so this notebook is reproducible standalone
df = orders.copy()
bad_mask = df['actual_delivery_min'] < 0
median_by_cat = df.loc[~bad_mask].groupby('food_category')['actual_delivery_min'].median()
df.loc[bad_mask, 'actual_delivery_min'] = df.loc[bad_mask, 'food_category'].map(median_by_cat)

rest_avg = df.groupby('restaurant_id')['rating'].transform('mean')
df['rating'] = df['rating'].fillna(rest_avg).fillna(df['rating'].mean()).round(1)

df['is_late'] = df['actual_delivery_min'] > df['promised_delivery_min']
df['order_month'] = df['order_timestamp'].dt.to_period('M').astype(str)
df['order_hour'] = df['order_timestamp'].dt.hour
df['day_of_week'] = df['order_timestamp'].dt.day_name()

print('Cleaned. Remaining nulls:', df.isna().sum().sum())

## 3. Revenue Trends

In [ ]:
monthly_revenue = df.groupby('order_month')['final_amount'].sum()
ax = monthly_revenue.plot(kind='line', marker='o', title='Monthly Revenue Trend')
ax.set_ylabel('Revenue (\u20b9)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
city_revenue = df.groupby('city')['final_amount'].sum().sort_values(ascending=False)
sns.barplot(x=city_revenue.values, y=city_revenue.index, hue=city_revenue.index, legend=False, palette='viridis')
plt.title('Revenue by City')
plt.xlabel('Revenue (\u20b9)')
plt.tight_layout()
plt.show()

In [ ]:
cat_revenue = df.groupby('food_category')['final_amount'].sum().sort_values(ascending=False)
sns.barplot(x=cat_revenue.values, y=cat_revenue.index, hue=cat_revenue.index, legend=False, palette='mako')
plt.title('Revenue by Food Category')
plt.xlabel('Revenue (\u20b9)')
plt.tight_layout()
plt.show()

## 4. Customer Behavior

In [ ]:
order_counts = df.groupby('customer_id').size()
repeat_rate = (order_counts >= 2).mean() * 100
print(f'Repeat purchase rate: {repeat_rate:.2f}%')

plt.figure()
sns.histplot(order_counts, bins=30)
plt.title('Distribution of Orders per Customer')
plt.xlabel('Number of Orders')
plt.tight_layout()
plt.show()

In [ ]:
clv = df.groupby('customer_id')['final_amount'].sum().sort_values(ascending=False)
print('Top 10 customers by lifetime value:')
print(clv.head(10))

plt.figure()
sns.histplot(clv, bins=40)
plt.title('Customer Lifetime Value Distribution')
plt.xlabel('Lifetime Value (\u20b9)')
plt.tight_layout()
plt.show()

## 5. Delivery Operations

In [ ]:
late_pct = df['is_late'].mean() * 100
print(f'Overall late-delivery rate: {late_pct:.2f}%')

late_by_city = df.groupby('city')['is_late'].mean().sort_values(ascending=False) * 100
sns.barplot(x=late_by_city.values, y=late_by_city.index, hue=late_by_city.index, legend=False, palette='rocket')
plt.title('Late Delivery % by City')
plt.xlabel('Late Delivery %')
plt.tight_layout()
plt.show()

In [ ]:
hourly_orders = df.groupby('order_hour').size()
sns.barplot(x=hourly_orders.index, y=hourly_orders.values, hue=hourly_orders.index, legend=False, palette='flare')
plt.title('Orders by Hour of Day (Peak Ordering Hours)')
plt.xlabel('Hour of Day')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
sns.scatterplot(data=df.sample(3000, random_state=1), x='actual_delivery_min', y='rating', alpha=0.3)
plt.title('Delivery Time vs Rating (sampled)')
plt.tight_layout()
plt.show()

print('Correlation (delivery time vs rating):', df['actual_delivery_min'].corr(df['rating']).round(3))

## 6. Key Takeaways

- Revenue is concentrated in a handful of top cities and food categories — see Section 3.
- Repeat customers make up a large share of orders despite being a minority of the customer base, which is why retention and CLV work matter more than pure acquisition.
- Late deliveries correlate with lower ratings, directly connecting operations performance to customer satisfaction and, downstream, retention.
- Peak ordering hours cluster around lunch and dinner, which should inform delivery partner staffing.

These findings are formalized as reusable SQL views in `sql/03_analysis_queries.sql` and visualized in the Power BI dashboard (`powerbi/POWERBI_GUIDE.md`).